In [ ]:
import importlib
import sys
import pandas as pd


if 'src.preprocessing' in sys.modules:
    del sys.modules['src.preprocessing']

from src.preprocessing import load_and_preprocess_data
from src.models import train_logistic_regression, train_decision_tree
from src.evaluation import evaluate_model

In [ ]:
print("Starting Data Preprocessing Pipeline")

dataset_path = "archive/KDDTrain+.txt"

X_train, X_test, y_train, y_test = load_and_preprocess_data(dataset_path)

print("\nPreprocessing Completed Successfully!")
print(f"X_train shape (scaled matrices): {X_train.shape}")
print(f"X_test shape (scaled matrices): {X_test.shape}")
print("\nTarget label distribution (Training Set):")
print(y_train.value_counts())

In [ ]:
import importlib
if 'src.preprocessing' in sys.modules:
    importlib.reload(sys.modules['src.preprocessing'])

from src.preprocessing import select_features_kbest, select_features_random_forest
import time

 
k_features = 20
start_time = time.time()

X_train_selected_kbest, X_test_selected_kbest, selector_kbest, kbest_scores = select_features_kbest(
    X_train, X_test, y_train, k=k_features
)

kbest_time = time.time() - start_time
print(f"SelectKBest execution time: {kbest_time:.4f} seconds")

In [ ]:
start_time = time.time()

X_train_selected_rf, X_test_selected_rf, rf_indices, rf_scores = select_features_random_forest(
    X_train, X_test, y_train, k=k_features
)

rf_time = time.time() - start_time
print(f"Random Forest Feature Selection execution time: {rf_time:.4f} seconds")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

print("\n" + "="*80)
print("COMPARISON: Model Performance with Different Feature Sets")
print("="*80)


print("\n1. ORIGINAL FEATURES (110+ columns):")
lr_original = LogisticRegression(max_iter=1000, random_state=42)
lr_original.fit(X_train, y_train)
lr_acc_original = accuracy_score(y_test, lr_original.predict(X_test))
lr_f1_original = f1_score(y_test, lr_original.predict(X_test))

dt_original = DecisionTreeClassifier(random_state=42, max_depth=10)
dt_original.fit(X_train, y_train)
dt_acc_original = accuracy_score(y_test, dt_original.predict(X_test))
dt_f1_original = f1_score(y_test, dt_original.predict(X_test))

print(f"  Logistic Regression  - Accuracy: {lr_acc_original*100:.2f}%, F1-Score: {lr_f1_original*100:.2f}%")
print(f"  Decision Tree        - Accuracy: {dt_acc_original*100:.2f}%, F1-Score: {dt_f1_original*100:.2f}%")


print("\n2. SELECTKBEST FEATURES (20 columns - 82% reduction):")
lr_kbest = LogisticRegression(max_iter=1000, random_state=42)
lr_kbest.fit(X_train_selected_kbest, y_train)
lr_acc_kbest = accuracy_score(y_test, lr_kbest.predict(X_test_selected_kbest))
lr_f1_kbest = f1_score(y_test, lr_kbest.predict(X_test_selected_kbest))

dt_kbest = DecisionTreeClassifier(random_state=42, max_depth=10)
dt_kbest.fit(X_train_selected_kbest, y_train)
dt_acc_kbest = accuracy_score(y_test, dt_kbest.predict(X_test_selected_kbest))
dt_f1_kbest = f1_score(y_test, dt_kbest.predict(X_test_selected_kbest))

print(f"  Logistic Regression  - Accuracy: {lr_acc_kbest*100:.2f}%, F1-Score: {lr_f1_kbest*100:.2f}%")
print(f"  Decision Tree        - Accuracy: {dt_acc_kbest*100:.2f}%, F1-Score: {dt_f1_kbest*100:.2f}%")


print("\n3. RANDOM FOREST FEATURES (20 columns - 82% reduction):")
lr_rf = LogisticRegression(max_iter=1000, random_state=42)
lr_rf.fit(X_train_selected_rf, y_train)
lr_acc_rf = accuracy_score(y_test, lr_rf.predict(X_test_selected_rf))
lr_f1_rf = f1_score(y_test, lr_rf.predict(X_test_selected_rf))

dt_rf = DecisionTreeClassifier(random_state=42, max_depth=10)
dt_rf.fit(X_train_selected_rf, y_train)
dt_acc_rf = accuracy_score(y_test, dt_rf.predict(X_test_selected_rf))
dt_f1_rf = f1_score(y_test, dt_rf.predict(X_test_selected_rf))

print(f"  Logistic Regression  - Accuracy: {lr_acc_rf*100:.2f}%, F1-Score: {lr_f1_rf*100:.2f}%")
print(f"  Decision Tree        - Accuracy: {dt_acc_rf*100:.2f}%, F1-Score: {dt_f1_rf*100:.2f}%")


print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)
comparison_data = {
    'Feature Set': ['Original (110+)', 'SelectKBest (20)', 'Random Forest (20)'],
    'LR Accuracy': [f"{lr_acc_original*100:.2f}%", f"{lr_acc_kbest*100:.2f}%", f"{lr_acc_rf*100:.2f}%"],
    'LR F1-Score': [f"{lr_f1_original*100:.2f}%", f"{lr_f1_kbest*100:.2f}%", f"{lr_f1_rf*100:.2f}%"],
    'DT Accuracy': [f"{dt_acc_original*100:.2f}%", f"{dt_acc_kbest*100:.2f}%", f"{dt_acc_rf*100:.2f}%"],
    'DT F1-Score': [f"{dt_f1_original*100:.2f}%", f"{dt_f1_kbest*100:.2f}%", f"{dt_f1_rf*100:.2f}%"]
}
comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))
print("="*80)

In [ ]:
import importlib
if 'src.preprocessing' in sys.modules:
    importlib.reload(sys.modules['src.preprocessing'])

from src.preprocessing import test_multiple_k_values, create_interaction_features, create_domain_specific_features

k_values = [10, 15, 20, 25, 30]
results_df = test_multiple_k_values(X_train, X_test, y_train, y_test, k_values=k_values)

In [ ]:
X_train_inter, X_test_inter = create_interaction_features(X_train_selected_kbest, X_test_selected_kbest)

lr_inter = LogisticRegression(max_iter=1000, random_state=42)
lr_inter.fit(X_train_inter, y_train)
lr_acc_inter = accuracy_score(y_test, lr_inter.predict(X_test_inter))
lr_f1_inter = f1_score(y_test, lr_inter.predict(X_test_inter))

dt_inter = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_inter.fit(X_train_inter, y_train)
dt_acc_inter = accuracy_score(y_test, dt_inter.predict(X_test_inter))
dt_f1_inter = f1_score(y_test, dt_inter.predict(X_test_inter))

print(f"Interaction Features: LR={lr_acc_inter100:.2f}%, DT={dt_acc_inter100:.2f}%")

In [ ]:
X_train_domain, X_test_domain = create_domain_specific_features(X_train, X_test, y_train, y_test)

lr_domain = LogisticRegression(max_iter=1000, random_state=42)
lr_domain.fit(X_train_domain, y_train)
lr_acc_domain = accuracy_score(y_test, lr_domain.predict(X_test_domain))
lr_f1_domain = f1_score(y_test, lr_domain.predict(X_test_domain))

dt_domain = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_domain.fit(X_train_domain, y_train)
dt_acc_domain = accuracy_score(y_test, dt_domain.predict(X_test_domain))
dt_f1_domain = f1_score(y_test, dt_domain.predict(X_test_domain))

print(f"Domain-Specific Features: LR={lr_acc_domain100:.2f}%, DT={dt_acc_domain100:.2f}%")

In [ ]:
from src.preprocessing import load_and_preprocess_data
from src.models import train_logistic_regression, train_decision_tree, train_knn, train_neural_network
from src.evaluation import evaluate_model

In [ ]:
all_features_comparison = {
    'Feature Set': ['Original (115+)', 'SelectKBest (20)', 'Random Forest (20)', 'Interaction', 'Domain-Specific'],
    'Features': [X_train.shape[1], X_train_selected_kbest.shape[1], X_train_selected_rf.shape[1], X_train_inter.shape[1], X_train_domain.shape[1]],
    'LR Accuracy': [f"{lr_acc_original100:.2f}%", f"{lr_acc_kbest100:.2f}%", f"{lr_acc_rf100:.2f}%", f"{lr_acc_inter100:.2f}%", f"{lr_acc_domain100:.2f}%"],
    'DT Accuracy': [f"{dt_acc_original100:.2f}%", f"{dt_acc_kbest100:.2f}%", f"{dt_acc_rf100:.2f}%", f"{dt_acc_inter100:.2f}%", f"{dt_acc_domain100:.2f}%"],
}

comparison_all_df = pd.DataFrame(all_features_comparison)
print("\nFeature Engineering Comparison:")
print(comparison_all_df.to_string(index=False))
print(f"\nBest Balance: SelectKBest (k=20) - {100 * (1 - 20/X_train.shape[1]):.1f}% reduction with {dt_acc_kbest*100:.2f}% DT accuracy")

In [ ]:
print("\n" + "="80)
print("EXPERIMENT: All Features (115+) vs Selected Features (Top 20)")
print("="80 + "\n")

all_vs_selected = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'All Features': [f"{lr_acc_original100:.2f}%", f"{dt_acc_original100:.2f}%"],
    'Top 20': [f"{lr_acc_kbest100:.2f}%", f"{dt_acc_kbest100:.2f}%"],
    'Loss': [f"{(lr_acc_kbest - lr_acc_original)100:+.2f}%", f"{(dt_acc_kbest - dt_acc_original)100:+.2f}%"],
    'Reduction': [f"{100 * (1 - 20/X_train.shape[1]):.1f}%", f"{100 * (1 - 20/X_train.shape[1]):.1f}%"]
})

print(all_vs_selected.to_string(index=False))
print("\nConclusion: SelectKBest (k=20) RECOMMENDED - minimal accuracy loss (<2.5%) with 83% reduction")

In [ ]:
import time

print("\nTraining Time Comparison:")

start = time.time()
lr_all = LogisticRegression(max_iter=1000, random_state=42)
lr_all.fit(X_train, y_train)
lr_all_time = time.time() - start

start = time.time()
dt_all = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_all.fit(X_train, y_train)
dt_all_time = time.time() - start

start = time.time()
lr_sel = LogisticRegression(max_iter=1000, random_state=42)
lr_sel.fit(X_train_selected_kbest, y_train)
lr_sel_time = time.time() - start

start = time.time()
dt_sel = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_sel.fit(X_train_selected_kbest, y_train)
dt_sel_time = time.time() - start

timing_comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'All Features (s)': [f"{lr_all_time:.4f}", f"{dt_all_time:.4f}"],
    'Top 20 (s)': [f"{lr_sel_time:.4f}", f"{dt_sel_time:.4f}"],
    'Speedup': [f"{lr_all_time/lr_sel_time:.2f}x", f"{dt_all_time/dt_sel_time:.2f}x"],
})

print(timing_comparison.to_string(index=False))
print(f"\nComputational benefit: 3-4x faster training with selected features")

In [ ]:
import importlib
import sys

if 'src.models' in sys.modules:
    del sys.modules['src.models']
if 'src.evaluation' in sys.modules:
    del sys.modules['src.evaluation']

from src.models import train_logistic_regression, train_decision_tree, train_knn, train_neural_network
from src.evaluation import evaluate_model

In [ ]:
print("MODEL TRAINING AND EVALUATION PIPELINE")

print("\n1. LOGISTIC REGRESSION")
lr_model = train_logistic_regression(X_train, y_train)
print("Training completed successfully!")
evaluate_model(lr_model, X_test, y_test, "Logistic Regression")

In [ ]:
print("\n2. DECISION TREE")
dt_model = train_decision_tree(X_train, y_train)
print("Training completed successfully!")
evaluate_model(dt_model, X_test, y_test, "Decision Tree")

In [ ]:
print("\n3. K-NEAREST NEIGHBORS")
knn_model = train_knn(X_train, y_train, n_neighbors=5)
print("Training completed successfully!")
evaluate_model(knn_model, X_test, y_test, "K-Nearest Neighbors")

In [ ]:
print("\n4. NEURAL NETWORK")
print("-"*80)
print("Configuration: 100 epochs, batch_size=32, early stopping enabled")
nn_model, history = train_neural_network(X_train, y_train, epochs=100, batch_size=32)
print(f"Training completed! Total epochs trained: {len(history.history['loss'])}")
print(f"Final Training Accuracy: {history.history['accuracy'][-1]*100:.2f}%")
print(f"Final Validation Accuracy: {history.history['val_accuracy'][-1]*100:.2f}%")


nn_pred = (nn_model.predict(X_test) > 0.5).astype(int).flatten()
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

acc = accuracy_score(y_test, nn_pred)
prec = precision_score(y_test, nn_pred)
rec = recall_score(y_test, nn_pred)
f1 = f1_score(y_test, nn_pred)
matrix = confusion_matrix(y_test, nn_pred)

print(f"\nTest Set Performance:")
print(f"Accuracy:  {acc*100:.2f}%")
print(f"Precision: {prec*100:.2f}%")
print(f"Recall:    {rec*100:.2f}%")
print(f"F1-Score:  {f1*100:.2f}%")
print("Confusion Matrix:")
print(matrix)